In [ ]:
#| default_exp _trainer

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
#| export
import torch
import matplotlib.pyplot as plt
from dreamer4._core import build_tiny_pinpad_dataset, make_tiny_dataloader
from dreamer4.dreamer4 import VideoTokenizer, DynamicsWorldModel
from dreamer4.trainers import VideoTokenizerTrainer, SimTrainer, cycle

In [ ]:
CPU = False
device = 'cuda' if not CPU else 'cpu'
dataset, env = build_tiny_pinpad_dataset(device=device, episode_length=100)

In [ ]:
tiny_loader = make_tiny_dataloader(dataset, n_items=1024, batch_size=16)
lpips_loss = 0.05
dim, dim_latent = 64, 64
tokenizer = VideoTokenizer(
    dim=dim,#16,
    # encoder_depth = 1, # I think the paper has a different depths and does the time blocks every n (8?) layers
    # decoder_depth = 1,
    # time_block_every = 1,
    dim_latent = dim_latent, #16,
    patch_size = 16, #32
    attn_dim_head = 16,
    num_latent_tokens = 4,
    lpips_loss_weight=lpips_loss,
).to(device)

world_model_and_policy = DynamicsWorldModel(
    video_tokenizer = tokenizer,
    dim = dim, #16,
    dim_latent = dim_latent, #16,
    max_steps = 64,
    num_tasks = 1,
    num_latent_tokens = 4, #1,
    # depth = 1,
    # time_block_every = 1,
    # num_spatial_tokens = 1,
    pred_orig_latent = True,
    num_discrete_actions = env.action_space.n,
    attn_dim_head = 16,
    prob_no_shortcut_train = 0.1,
    num_residual_streams = 1
).to(device)

In [ ]:
wm_trainer = VideoTokenizerTrainer(
    model = tokenizer,
    dataset = dataset,
    cpu = CPU,
    num_train_steps=30
)

losses = wm_trainer.forward()

In [ ]:
trainer = SimTrainer(
    world_model_and_policy,
    batch_size = 32,
    cpu = CPU
)
trainer(env, num_episodes = 1, env_is_vectorized = False)

In [ ]:
trainer.eval_episodes(
    env=env,
    tokenizer=tokenizer,
    num_episodes=1,
    max_steps=30,
    env_is_vectorized=False,
    max_time_to_show=8,
)

In [ ]:
ep_length = 100
n = 32
dataset, env = build_tiny_pinpad_dataset(device=device, episode_length=ep_length,n=n)

pretrain_steps = 5000
wm_pretrainer = VideoTokenizerTrainer(
    model = tokenizer,
    dataset = dataset,
    cpu = CPU,
    num_train_steps=pretrain_steps
)

wm_trainer = VideoTokenizerTrainer(
    model = tokenizer,
    dataset = dataset,
    cpu = CPU,
    num_train_steps=ep_length
)

trainer = SimTrainer(
    world_model_and_policy,
    batch_size = 32,
    cpu = CPU
)

In [ ]:

# All together now
if debug:=False:
    epochs = 1
    pretrain=True; pretrain_eps=10; policy_pretrain_eps=2
    train_wm=True
else:
    epochs = 100
    pretrain=True; pretrain_eps=1000; policy_pretrain_eps=100
    train_wm=True

pretrain=False

# Reset the policy
# world_model_and_policy = DynamicsWorldModel(
#     video_tokenizer = tokenizer,
#     dim = dim, #16,
#     dim_latent = dim_latent, #16,
#     max_steps = 64,
#     num_tasks = 1,
#     num_latent_tokens = 4, #1,
#     # depth = 1,
#     # time_block_every = 1,
#     # num_spatial_tokens = 1,
#     pred_orig_latent = True,
#     num_discrete_actions = env.action_space.n,
#     attn_dim_head = 16,
#     prob_no_shortcut_train = 0.1,
#     num_residual_streams = 1
# ).to(device)

trainer = SimTrainer(
    world_model_and_policy,
    batch_size = 32,
    cpu = CPU,
    epochs=epochs,
).to(device)

for epoch in range(epochs):
    if pretrain:
        print(f'Initializing pretraining...')
        pretrain = False
        if train_wm: wm_pretrainer.forward()
        trainer(env, num_episodes=policy_pretrain_eps, max_experiences_before_learn=ep_length, env_is_vectorized = False)
    else:
        if train_wm: losses = wm_trainer.forward()
        trainer(env, num_episodes = 1, env_is_vectorized = False)

    if epoch % 10 == 0:
        trainer.eval_episodes(
            env=env,
            tokenizer=tokenizer,
            num_episodes=1,
            max_steps=ep_length, 
            env_is_vectorized=False,
            max_time_to_show=8,
        )

In [ ]:
eval_exp = trainer.eval_episodes(
    env=env,
    tokenizer=tokenizer,
    num_episodes=100,
    max_steps=ep_length,
    env_is_vectorized=False,
    max_time_to_show=8,
    show=False,
)

In [ ]:
all_actions = [entry.actions[0].cpu().numpy()[0].squeeze() for entry in eval_exp]
all_rewards = [entry.rewards[0].cpu().numpy().squeeze() for entry in eval_exp]

In [ ]:
f'Mean reward {np.sum([np.sum(entry) for entry in all_rewards]) / len(all_rewards):+1.2}'

In [ ]:
import numpy as np
rewards = np.concatenate(all_rewards); rewards.shape
actions = np.concatenate(all_actions); actions.shape
plt.hist(actions)
plt.hist(rewards)

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()